# Level 3: Core Numerical Methods Engine

**Course:** ICS 2207 — Scientific Computing  
**Project:** HydroSense-Kenya  
**Objective:** Implement core numerical methods from scratch and apply them to irrigation decisions.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.numerical_methods import (
    bisection, newton_raphson, secant,
    forward_difference, backward_difference, central_difference,
    trapezoidal_rule, simpsons_rule,
    gaussian_elimination, lu_decomposition, lu_solve
)

weather = pd.read_csv('../data/raw/weather_daily.csv', na_values=['NA', ''])
soil    = pd.read_csv('../data/raw/soil_sensor_data.csv', na_values=['NA', ''])
params  = pd.read_csv('../data/raw/crop_zone_parameters.csv')

weather['date'] = pd.to_datetime(weather['date'])
soil['timestamp'] = pd.to_datetime(soil['timestamp'])

---

## 1 & 2. Root Finding — Irrigation Amount to Reach Target Moisture

We define $f(I)$ as the difference between next-day soil moisture (given irrigation $I$) and the target moisture. Finding the root $f(I)=0$ tells us exactly how much water to apply.

$$f(I) = S_t + R_t + I - ET_t - D_t(I) - S_{target}$$

where $D_t(I) = d_c \cdot \max(0,\; S_t + R_t + I - ET_t - FC)$

In [ ]:
# Set up the problem for Zone_A on a specific day
zone_a = params[params['zone_id'] == 'Zone_A'].iloc[0]
S_t = 25.0           # current soil moisture (%)
R_t = 2.0            # today's rainfall (mm-equiv)
ET_t = 4.5           # today's ET
fc = zone_a['field_capacity_pct']       # 41
dc = zone_a['drainage_coefficient']     # 0.18
S_target = zone_a['target_moisture_pct'] # 33

def f_irrigation(I):
    """f(I) = next-day moisture - target. Root gives required irrigation."""
    S_interim = S_t + R_t + I - ET_t
    D = dc * max(0.0, S_interim - fc)
    S_next = S_interim - D
    return S_next - S_target

def f_irrigation_prime(I):
    """Derivative of f(I) with respect to I."""
    S_interim = S_t + R_t + I - ET_t
    if S_interim > fc:
        return 1.0 - dc  # drainage kicks in
    return 1.0

print(f'Zone A parameters: S_t={S_t}, R={R_t}, ET={ET_t}, FC={fc}, dc={dc}, target={S_target}')
print(f'f(0)  = {f_irrigation(0):.4f}  (negative → moisture below target)')
print(f'f(20) = {f_irrigation(20):.4f}  (positive → moisture above target)')

In [ ]:
# Apply all three root-finding methods
result_bisect = bisection(f_irrigation, 0, 20, tol=1e-6)
result_newton = newton_raphson(f_irrigation, f_irrigation_prime, 10.0, tol=1e-6)
result_secant = secant(f_irrigation, 0, 20, tol=1e-6)

print(f"{'Method':<18} {'Root (mm)':<14} {'Iterations':<12} {'Final Error':<14} {'Converged'}")
print('=' * 68)
for name, res in [('Bisection', result_bisect), ('Newton-Raphson', result_newton), ('Secant', result_secant)]:
    print(f"{name:<18} {res['root']:<14.6f} {res['iterations']:<12} {res['error']:<14.2e} {res['converged']}")

---

## 3. Convergence Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for name, res, color, marker in [
    ('Bisection', result_bisect, '#3b82f6', 'o'),
    ('Newton-Raphson', result_newton, '#ef4444', 's'),
    ('Secant', result_secant, '#22c55e', '^')
]:
    iters = [h['iteration'] for h in res['history']]
    errors = [h['error'] for h in res['history']]
    ax.semilogy(iters, errors, f'{marker}-', color=color, linewidth=2,
                markersize=7, label=f"{name} ({res['iterations']} iters)")

ax.axhline(1e-6, linestyle='--', color='gray', alpha=0.5, label='Tolerance (1e-6)')
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Error (log scale)', fontsize=12)
ax.set_title('Root-Finding Convergence Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/level3_convergence.png', dpi=150, bbox_inches='tight')
plt.show()

**Interpretation:** Newton-Raphson converges fastest (quadratic convergence) since it uses derivative information. Secant is nearly as fast without requiring the derivative. Bisection is the slowest (linear convergence) but is guaranteed to converge when the bracket is valid.

---

## 4. Finite Differences — Rate of Soil-Moisture Change

We estimate $dS/dt$ for each zone using forward, backward, and central differences with $h = 1$ day.

In [ ]:
h = 1.0  # 1-day time step

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Rate of Soil-Moisture Change by Zone (Finite Differences)',
             fontsize=14, fontweight='bold')

colors = {'Zone_A': '#22c55e', 'Zone_B': '#f59e0b', 'Zone_C': '#8b5cf6'}
zone_names = {'Zone_A': 'Zone A (Tomato)', 'Zone_B': 'Zone B (Kale)', 'Zone_C': 'Zone C (Maize)'}

for idx, zone_id in enumerate(['Zone_A', 'Zone_B', 'Zone_C']):
    zd = soil[soil['zone_id'] == zone_id].sort_values('timestamp').reset_index(drop=True)
    moisture = zd['soil_moisture_pct'].values
    dates = zd['timestamp'].values
    n = len(moisture)

    fwd = forward_difference(moisture, h)
    bwd = backward_difference(moisture, h)
    ctr = central_difference(moisture, h)

    ax = axes[idx]
    ax.plot(dates[:-1], fwd, 'o-', markersize=3, linewidth=1.2, color='#3b82f6', label='Forward', alpha=0.8)
    ax.plot(dates[1:], bwd, 's-', markersize=3, linewidth=1.2, color='#ef4444', label='Backward', alpha=0.8)
    ax.plot(dates[1:-1], ctr, '^-', markersize=3, linewidth=1.2, color='#22c55e', label='Central', alpha=0.8)
    ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
    ax.set_ylabel('dS/dt (%/day)')
    ax.set_title(zone_names[zone_id], loc='left', fontsize=11)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../reports/level3_finite_differences.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Numerical comparison table for one zone
zd = soil[soil['zone_id'] == 'Zone_A'].sort_values('timestamp').reset_index(drop=True)
moisture = zd['soil_moisture_pct'].values

fwd = forward_difference(moisture, h)
bwd = backward_difference(moisture, h)
ctr = central_difference(moisture, h)

print(f"{'Day':<6} {'Forward':<12} {'Backward':<12} {'Central':<12}")
print('=' * 42)
for i in range(1, min(10, len(ctr) + 1)):
    print(f"{i+1:<6} {fwd[i]:<12.4f} {bwd[i]:<12.4f} {ctr[i-1] if i-1 < len(ctr) else 'N/A':<12}")

**Note:** Central differences are $O(h^2)$ accurate while forward/backward are $O(h)$. With $h=1$ day (our only option), the central estimate averages the adjacent days' slopes and tends to be smoother.

---

## 5. Numerical Integration — Cumulative Water Deficit

Water deficit = how much soil moisture falls below the minimum threshold over time. We compute the deficit curve and integrate it using both trapezoidal and Simpson's rules.

In [ ]:
zone_ids = ['Zone_A', 'Zone_B', 'Zone_C']
integration_results = []

for zone_id in zone_ids:
    zp = params[params['zone_id'] == zone_id].iloc[0]
    zd = soil[soil['zone_id'] == zone_id].sort_values('timestamp').reset_index(drop=True)
    moisture = zd['soil_moisture_pct'].fillna(method='ffill').values
    min_m = zp['min_moisture_pct']

    # Deficit: how far below minimum (0 if above)
    deficit = np.maximum(0, min_m - moisture)

    # Ensure even number of intervals for Simpson's rule
    if len(deficit) % 2 == 0:
        deficit_simpson = deficit[:-1]  # trim to odd number of points
    else:
        deficit_simpson = deficit

    trap = trapezoidal_rule(deficit, h=1.0)
    simp = simpsons_rule(deficit_simpson, h=1.0)

    integration_results.append({
        'Zone': zone_id,
        'Crop': zp['crop_type'],
        'Days with deficit': int(np.sum(deficit > 0)),
        'Max deficit (%)': f"{deficit.max():.2f}",
        'Trapezoidal (%-days)': f"{trap:.2f}",
        "Simpson's (%-days)": f"{simp:.2f}",
        'Difference': f"{abs(trap - simp):.4f}"
    })

int_df = pd.DataFrame(integration_results)
print('Cumulative Water Deficit — Integration Comparison')
print('=' * 80)
int_df

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
fig.suptitle('Water Deficit Over Time (moisture below minimum threshold)',
             fontsize=14, fontweight='bold')

for idx, zone_id in enumerate(zone_ids):
    zp = params[params['zone_id'] == zone_id].iloc[0]
    zd = soil[soil['zone_id'] == zone_id].sort_values('timestamp').reset_index(drop=True)
    moisture = zd['soil_moisture_pct'].fillna(method='ffill').values
    dates = zd['timestamp'].values
    min_m = zp['min_moisture_pct']
    deficit = np.maximum(0, min_m - moisture)

    ax = axes[idx]
    ax.fill_between(dates, deficit, alpha=0.4, color=colors[zone_id])
    ax.plot(dates, deficit, '-', color=colors[zone_id], linewidth=1.5)
    ax.set_ylabel('Deficit (%)')
    ax.set_title(f"{zone_names[zone_id]} — min threshold = {min_m}%", loc='left', fontsize=11)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Date')
plt.tight_layout()
plt.savefig('../reports/level3_water_deficit.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. Linear Systems — Three-Zone Water Allocation

We formulate a system $Ax = b$ to determine the optimal water allocation across three zones.

**Problem:** Given a total daily water budget, allocate irrigation water $[x_A, x_B, x_C]$ such that:
1. Total allocation equals available supply
2. Each zone's allocation is proportional to its area-weighted deficit
3. Pump capacity constraints link the zones

$$\begin{cases}
x_A + x_B + x_C = W_{total} \\
\frac{x_A}{A_A} - \frac{x_B}{A_B} = d_A - d_B \\
\frac{x_B}{A_B} - \frac{x_C}{A_C} = d_B - d_C
\end{cases}$$

In [ ]:
# Zone areas and current deficits (from last day of data)
areas = params.set_index('zone_id')['area_m2']
A_A, A_B, A_C = areas['Zone_A'], areas['Zone_B'], areas['Zone_C']  # 120, 90, 180

# Compute current deficit for each zone (last observation)
deficits = {}
for zone_id in zone_ids:
    zp = params[params['zone_id'] == zone_id].iloc[0]
    zd = soil[soil['zone_id'] == zone_id].sort_values('timestamp')
    last_moisture = zd['soil_moisture_pct'].iloc[-1]
    deficits[zone_id] = max(0, zp['target_moisture_pct'] - last_moisture)

d_A = deficits['Zone_A']
d_B = deficits['Zone_B']
d_C = deficits['Zone_C']

W_total = 50.0  # total available water (mm-equivalent)

print(f'Zone deficits: A={d_A:.1f}%, B={d_B:.1f}%, C={d_C:.1f}%')
print(f'Zone areas:    A={A_A} m², B={A_B} m², C={A_C} m²')
print(f'Total water budget: {W_total} mm-equiv')

In [ ]:
# Build the coefficient matrix and RHS vector
A = np.array([
    [1.0,       1.0,       1.0],         # total constraint
    [1.0/A_A,  -1.0/A_B,   0.0],         # proportional equity A vs B
    [0.0,       1.0/A_B,  -1.0/A_C]      # proportional equity B vs C
])

b = np.array([W_total, d_A - d_B, d_B - d_C])

print('Coefficient matrix A:')
print(A)
print(f'\nRHS vector b: {b}')

In [ ]:
# Solve with Gaussian Elimination
x_gauss = gaussian_elimination(A, b)
print('Solution via Gaussian Elimination:')
print(f'  Zone_A: {x_gauss[0]:.4f} mm')
print(f'  Zone_B: {x_gauss[1]:.4f} mm')
print(f'  Zone_C: {x_gauss[2]:.4f} mm')
print(f'  Sum:    {x_gauss.sum():.4f} (should be {W_total})')

In [ ]:
# Solve with LU Decomposition
L, U, P = lu_decomposition(A)
x_lu = lu_solve(L, U, P, b)
print('Solution via LU Decomposition:')
print(f'  Zone_A: {x_lu[0]:.4f} mm')
print(f'  Zone_B: {x_lu[1]:.4f} mm')
print(f'  Zone_C: {x_lu[2]:.4f} mm')

# Verify against NumPy
x_numpy = np.linalg.solve(A, b)
print(f'\nMax difference from np.linalg.solve:')
print(f'  Gauss: {np.max(np.abs(x_gauss - x_numpy)):.2e}')
print(f'  LU:    {np.max(np.abs(x_lu - x_numpy)):.2e}')

In [ ]:
# Visualize the allocation
fig, ax = plt.subplots(figsize=(8, 5))
zone_labels = ['Zone A\n(Tomato)', 'Zone B\n(Kale)', 'Zone C\n(Maize)']
bar_colors = ['#22c55e', '#f59e0b', '#8b5cf6']

bars = ax.bar(zone_labels, x_gauss, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, x_gauss):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.2f} mm', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Water Allocation (mm-equiv)', fontsize=12)
ax.set_title(f'Three-Zone Water Allocation (Total = {W_total} mm)',
             fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/level3_water_allocation.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretation

The linear system allocates more water to zones with larger area-weighted deficits. Both Gaussian elimination and LU decomposition produce the same solution, verified against NumPy's `linalg.solve`. The allocation respects the total water budget constraint and distributes water proportionally to each zone's need relative to its cultivated area.

---

*End of Level 3 — Core Numerical Methods Engine.*